#  Player Performance Prediction
### Regression — Predicting Next-Season Goals & Assists

---

> **Algorithm:** Random Forest Regressor with GridSearchCV  
> **Dataset:** Top 5 European Leagues, Seasons 2017–2024  
> **Targets:** `goals_next` · `assists_next`

---

##  Problem Statement

Predicting a player's future output is one of the most valuable tasks in football analytics. Scouts, coaches, and data analysts rely on performance forecasts to make transfer decisions, set expectations, and plan tactical systems.

This notebook trains two separate **Random Forest regressors** — one predicting a player's goals next season and one predicting assists — using career statistics from the Top 5 European leagues (Premier League, La Liga, Bundesliga, Serie A, Ligue 1) spanning 2017 to 2024.

**Key design choice:** We use a **time-aware train/test split** (train on seasons before 2023/24, test on 2023/24) to avoid data leakage and simulate a realistic forecasting scenario.

###  Notebook Structure
1. Imports & Configuration
2. Dataset Overview
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Exploratory Data Analysis
6. Model Building — Goals Predictor
7. Model Building — Assists Predictor
8. Model Comparison
9. Key Insights & Conclusion

---
## 1. 📦 Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Global settings 
RANDOM_STATE       = 42
TOP_N_FEATURES     = 25       # number of features selected by correlation
TRAIN_CUTOFF       = 2324     # seasons < 2324 used for training
TARGETS            = ['goals_next', 'assists_next']

np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120})

print(' Libraries loaded.')

---
## 2. 📂 Dataset Overview

The dataset contains season-level performance data for players across the top five European football leagues from 2017 to 2024. Each row represents one player-season.

Key columns include:
- **`Performance_Gls`** — Goals scored
- **`Performance_Ast`** — Assists
- **`Expected_xG`** — Expected goals (shot quality)
- **`Playing Time_90s`** — Matches played in 90-minute units
- **`Progression_PrgP`** — Progressive passes
- **`pos_`** — Playing position

In [ ]:
# ── Load data 
# Note: file uses semicolon delimiter and comma as decimal separator
df_raw = pd.read_csv(
    '/kaggle/input/Top5_League_Players_2017to2024_dataset.csv',
    sep=';', decimal=','
)

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
display(df_raw.head())

In [ ]:
print('Columns:', df_raw.columns.tolist())
print(f'\nSeasons covered: {sorted(df_raw["season"].unique())}')
print(f'Unique players : {df_raw["player"].nunique():,}')
print(f'Leagues        : {df_raw["league"].unique()}')

In [ ]:
# Summary statistics for key performance columns
perf_cols = ['Performance_Gls', 'Performance_Ast', 'Expected_xG',
             'Playing Time_90s', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast']
display(df_raw[perf_cols].describe())

---
## 3.  Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# Sort chronologically per player — essential before any temporal shift
df = df.sort_values(['player', 'season'])

print(f'Records before cleaning: {len(df):,}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

---
## 4.  Feature Engineering

### 4.1 Temporal Targets (Shifted by One Season)

To predict **next season's** performance, we shift each player's goals and assists forward by one row within their career history. This is the standard approach for sports time-series forecasting.

>  After shifting, the last season for every player will have a `NaN` target (no future exists yet). These rows are dropped.

In [ ]:
# ── Shift targets one season forward per player 
df['goals_next']   = df.groupby('player')['Performance_Gls'].shift(-1)
df['assists_next'] = df.groupby('player')['Performance_Ast'].shift(-1)

# ── Goal trend: current minus previous season (momentum signal) 
df['goals_trend'] = (
    df['Performance_Gls']
    - df.groupby('player')['Performance_Gls'].shift(1)
)

# ── Drop rows where target is NaN (last season per player) 
n_before = len(df)
df = df.dropna(axis=0)
print(f'Rows dropped (NaN targets): {n_before - len(df):,}')
print(f'Final dataset size: {len(df):,} player-seasons')

### 4.2 Position Encoding

Player positions are mapped to an **ordinal encoding** that reflects attacking output tendency:
- `GK = 1` → `DF = 2` → `MF = 3` → `FW = 4`

Hybrid positions (e.g., `FW,MF`) are assigned to the higher-attacking role.

In [ ]:
position_map = {
    'GK': 1,
    'DF': 2,
    'MF': 3, 'MF,DF': 3, 'DF,MF': 3,
    'FW': 4, 'FW,MF': 4, 'MF,FW': 4,
             'FW,DF': 4, 'DF,FW': 4
}

df['position_encoded'] = df['pos_'].map(position_map)

print('Position mapping:')
for pos, code in sorted(position_map.items(), key=lambda x: x[1]):
    print(f'  {pos:8s} → {code}')

# Save encoding for inference use
joblib.dump(position_map, 'position_encoding.pkl')
print('\n Position encoding saved.')

In [ ]:
# ── Drop non-numeric identifier columns 
model_df = df.drop(columns=['player', 'pos_', 'league', 'team', 'nation_'])

print(f'Modelling dataset: {model_df.shape[0]:,} rows × {model_df.shape[1]} columns')

---
## 5. 📊 Exploratory Data Analysis

### 5.1 Goals & Assists Trends Across Seasons

In [ ]:
season_avg = df.groupby('season')[['Performance_Gls', 'Performance_Ast']].mean()

fig, ax = plt.subplots(figsize=(11, 5))
season_avg['Performance_Gls'].plot(ax=ax, marker='o', linewidth=2.5,
                                    color='#d32f2f', label='Avg Goals / Player')
season_avg['Performance_Ast'].plot(ax=ax, marker='s', linewidth=2.5,
                                    color='#1565c0', linestyle='--', label='Avg Assists / Player')
ax.set_title('Average Goals & Assists Per Player — Top 5 Leagues (2017–2024)', fontsize=13)
ax.set_xlabel('Season')
ax.set_ylabel('Average per Player')
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 Goals Distribution by Position

In [ ]:
pos_order  = ['GK', 'DF', 'MF', 'FW']
pos_labels = ['GK', 'DF', 'MF', 'FW']

plot_df = df[df['pos_'].isin(pos_order)].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=plot_df, x='pos_', y='Performance_Gls',
            order=pos_order, palette='Reds', ax=axes[0])
axes[0].set_title('Goals Distribution by Position', fontsize=12)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Goals')

sns.boxplot(data=plot_df, x='pos_', y='Performance_Ast',
            order=pos_order, palette='Blues', ax=axes[1])
axes[1].set_title('Assists Distribution by Position', fontsize=12)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Assists')

plt.tight_layout()
plt.show()

### 5.3 xG vs Actual Goals (Shot Quality Analysis)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sample = df.sample(min(1500, len(df)), random_state=RANDOM_STATE)
ax.scatter(sample['Expected_xG'], sample['Performance_Gls'],
           alpha=0.3, color='#d32f2f', edgecolors='white', linewidths=0.3, s=25)
lim = max(sample['Expected_xG'].max(), sample['Performance_Gls'].max())
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='xG = Goals (expected line)')
ax.set_xlabel('Expected Goals (xG)')
ax.set_ylabel('Actual Goals')
ax.set_title('Expected Goals vs Actual Goals\n(above line = overperformer, below = underperformer)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

corr_xg_goals = df[['Expected_xG', 'Performance_Gls']].corr().iloc[0, 1]
print(f'Pearson correlation (xG vs Goals): {corr_xg_goals:.4f}')

---
## 6.  Model Building — Goals & Assists Predictors

We train both models using the same pipeline and grid search. Results are stored in a dictionary for final comparison.

### 6.1 Time-Aware Train / Test Split

Unlike random splits, we train on all seasons before 2023/24 and test on 2023/24. This mirrors a real-world scenario: _"given everything I know up to this season, predict next season."_

In [ ]:
results = {}   # store evaluation metrics for comparison
models  = {}   # store best model objects

train_mask = model_df['season'] < TRAIN_CUTOFF
test_mask  = model_df['season'] >= TRAIN_CUTOFF

print(f'Training seasons : {model_df[train_mask]["season"].unique()}')
print(f'Test season      : {model_df[test_mask]["season"].unique()}')
print(f'Train samples    : {train_mask.sum():,}')
print(f'Test  samples    : {test_mask.sum():,}')

### 6.2 Correlation-Based Feature Selection

For each target we select the **top 25 features by absolute correlation**. This step reduces noise and speeds up grid search without sacrificing meaningful signal.

In [ ]:
for target in TARGETS:
    print(f'\n{"="*65}')
    print(f'  TARGET: {target.upper()}')
    print(f'{"="*65}')

    # ── Correlation-based feature selection 
    corr_vals    = model_df.corr()[target].abs().sort_values(ascending=False)
    top_features = corr_vals.head(TOP_N_FEATURES).index.tolist()
    selected_df  = model_df[top_features + ['season']]

    # ── Heatmap of top correlations 
    corr_top = selected_df.drop(columns=['season']).corr()[target].abs().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(5, 9))
    sns.heatmap(corr_top.to_frame(), annot=True, fmt='.3f', cmap='coolwarm',
                linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.5})
    ax.set_title(f'|Correlation| with {target}', fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── Time-aware split 
    train_sel = selected_df['season'] < TRAIN_CUTOFF
    test_sel  = selected_df['season'] >= TRAIN_CUTOFF

    X_train = selected_df[train_sel].drop(columns=[target, 'season'])
    X_test  = selected_df[test_sel].drop(columns=[target, 'season'])
    y_train = selected_df[train_sel][target]
    y_test  = selected_df[test_sel][target]

    # ── Pipeline + GridSearchCV 
    pipe = Pipeline([('model', RandomForestRegressor(random_state=RANDOM_STATE))])

    param_grid = {
        'model__n_estimators':      [150, 200, 300],
        'model__max_depth':         [8, 12, None],
        'model__min_samples_split': [2, 5]
    }

    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    models[target] = best_model

    print(f'Best Params : {grid.best_params_}')
    print(f'Best CV R²  : {grid.best_score_:.4f}')

    # ── Evaluation metrics 
    y_pred_test  = best_model.predict(X_test)
    y_pred_train = best_model.predict(X_train)

    r2_test  = r2_score(y_test,  y_pred_test)
    r2_train = r2_score(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    results[target] = {
        'r2_train': r2_train, 'r2_test': r2_test,
        'mae': mae_test, 'rmse': rmse_test,
        'best_params': grid.best_params_,
        'cv_r2': grid.best_score_
    }

    print(f'Train R²  : {r2_train:.4f}')
    print(f'Test  R²  : {r2_test:.4f}')
    print(f'Test  MAE : {mae_test:.4f}')
    print(f'Test  RMSE: {rmse_test:.4f}')

    # ── Predicted vs Actual 
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(y_test, y_pred_test, alpha=0.35, color='#1565c0',
                    edgecolors='white', linewidths=0.3, s=20)
    lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
    axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')
    axes[0].set_title(f'Actual vs Predicted — {target}\nR² = {r2_test:.3f}')
    axes[0].legend(fontsize=9)

    # Feature importance
    feat_imp = pd.Series(
        best_model.named_steps['model'].feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False).head(12)

    feat_imp.plot(kind='barh', ax=axes[1], color='#2e7d32', edgecolor='white')
    axes[1].invert_yaxis()
    axes[1].set_title(f'Top-12 Feature Importances — {target}')
    axes[1].set_xlabel('Importance')

    plt.suptitle(f'Model Results: {target}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Residual distribution 
    residuals = y_test - y_pred_test
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(residuals, bins=40, color='#4a148c', edgecolor='white', alpha=0.8)
    ax.axvline(x=0, color='red', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.set_title(f'Residual Distribution — {target}')
    plt.tight_layout()
    plt.show()

    # ── Save model and processed dataset 
    joblib.dump(best_model, f'performance_prediction_{target}_model.pkl')
    selected_df.to_csv(f'performance_pred_{target}_dataset.csv', index=False)
    print(f'✅ Saved model  → performance_prediction_{target}_model.pkl')
    print(f'✅ Saved dataset → performance_pred_{target}_dataset.csv')

---
## 7. 📊 Model Comparison

Side-by-side summary of both prediction tasks.

In [ ]:
comparison_df = pd.DataFrame(results).T.reset_index()
comparison_df.columns = ['Target', 'Train R²', 'Test R²', 'Test MAE', 'Test RMSE', 'Best Params', 'CV R²']

display(comparison_df[['Target', 'Train R²', 'Test R²', 'Test MAE', 'Test RMSE', 'CV R²']]
        .style
        .background_gradient(subset=['Test R²'], cmap='YlGn')
        .format({'Train R²': '{:.4f}', 'Test R²': '{:.4f}',
                 'Test MAE': '{:.4f}', 'Test RMSE': '{:.4f}', 'CV R²': '{:.4f}'})
        .set_caption('Performance Prediction — Model Summary')
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
w = 0.3
ax.bar(x - w/2, [results[t]['r2_train'] for t in TARGETS], w,
       label='Train R²', color='#1565c0', alpha=0.85)
ax.bar(x + w/2, [results[t]['r2_test']  for t in TARGETS], w,
       label='Test R²',  color='#d32f2f', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(['Goals Next Season', 'Assists Next Season'], fontsize=11)
ax.set_ylabel('R² Score')
ax.set_ylim(0, 1.05)
ax.set_title('Train vs Test R² — Goals & Assists Models', fontsize=13)
ax.legend()
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## 8.  Key Insights & Conclusion

### Goals model findings:

| Insight | Detail |
|---------|--------|
| **xG is king** | Expected goals and shots per 90 are the top importance features, confirming that shot quality matters more than volume |
| **Goals trend matters** | Season-on-season momentum (`goals_trend`) adds meaningful signal beyond raw averages |
| **Position is a multiplier** | Position encoding reflects the strong positional bias in goal scoring |
| **Playing time matters less than efficiency** | Per-90 metrics outperform cumulative totals in feature importance |

### Assists model findings:

| Insight | Detail |
|---------|--------|
| **Key passes dominate** | Assists are best predicted by creative playmaking metrics (key passes, progressive passes) |
| **xA mirrors xG's role** | Expected assists (xA) carries the same predictive power for assists that xG does for goals |
| **Assists harder to predict than goals** | Lower R² for assists reflects greater randomness (a pass can only become an assist if the recipient finishes) |

### Methodological takeaways:
- **Time-aware splitting** is essential for sports prediction — random splits leak future season data into training
- **Correlation-based feature selection** (top 25) reduces noise without requiring PCA
- **Random Forest** handles non-linear position × volume × quality interactions naturally

### Future improvements:
- Experiment with XGBoost/LightGBM for potential R² gains
- Train **position-specific sub-models** (e.g., a striker-only goals model)
- Frame as a sequence learning problem using LSTMs over career trajectories
- Add opponent difficulty features (xG against, league position of opponents faced)